# 05. Deduplication (중복 이미지 제거)
- dHash/pHash 알고리즘으로 Train/Test 간 중복 이미지 탐지
- 데이터 누수(Data Leakage) 방지
- proposal 2장 품질관리 항목 구현

In [ ]:
# ============================================================
# 패키지 설치 (최초 1회)
# ============================================================
# !pip install imagehash

In [ ]:
# ============================================================
# 라이브러리 import
# ============================================================
import os
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
from PIL import Image
from tqdm.notebook import tqdm
import imagehash

# 한글 폰트
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("✅ 라이브러리 로드 완료")

In [ ]:
# ============================================================
# 경로 설정
# ============================================================
SHARD_ROOT  = "../data/train"
SHARD_NUMS  = [0, 1]   # shard_0 + shard_1

# 두 shard의 labels.csv를 합쳐서 로드
dfs = []
for i in SHARD_NUMS:
    shard_path = os.path.join(SHARD_ROOT, f"shard_{i}")
    df = pd.read_csv(os.path.join(shard_path, "labels.csv"), index_col=0)
    df["shard_name"] = f"shard_{i}"
    dfs.append(df)

labels_df = pd.concat(dfs, ignore_index=True)
print(f"전체 이미지 수: {len(labels_df):,}장")
print(labels_df["label"].value_counts().rename({0: "Real", 1: "AI Generated"}))


In [ ]:
# ============================================================
# 해시 계산 함수 정의
# dHash: 차분 해시 (경계선/엣지 기반)
# pHash: 인지 해시 (DCT 주파수 기반, 더 정밀)
# ============================================================
def compute_hashes(img_dir: str, df: pd.DataFrame, hash_size: int = 8):
    """
    이미지별 dHash + pHash 계산

    Args:
        img_dir  : 이미지 폴더 경로
        df       : labels.csv DataFrame
        hash_size: 해시 크기 (클수록 정밀, 느림)

    Returns:
        hash_df: image_name / dhash / phash 컬럼 DataFrame
    """
    results = []
    errors  = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="해시 계산 중"):
        img_path = os.path.join(img_dir, row["image_name"])
        try:
            img = Image.open(img_path).convert("RGB")
            d_hash = str(imagehash.dhash(img, hash_size=hash_size))   # 차분 해시
            p_hash = str(imagehash.phash(img, hash_size=hash_size))   # 인지 해시
            results.append({
                "image_name": row["image_name"],
                "label"     : row["label"],
                "dhash"     : d_hash,
                "phash"     : p_hash
            })
        except Exception as e:
            errors.append(row["image_name"])

    if errors:
        print(f"⚠️  로드 실패 이미지: {len(errors)}장")

    return pd.DataFrame(results)

In [ ]:
# ============================================================
# 1:1 균형 샘플링 기준으로 해시 계산
# Real 전체(36,069장) : AI 36,069장 → 총 72,138장
# ============================================================
real_df = labels_df[labels_df["label"] == 0]          # Real 전체
n_each  = len(real_df)                                  # Real 수 기준으로 AI도 맞춤
fake_df = labels_df[labels_df["label"] == 1].sample(n=n_each, random_state=42)
sample_df = pd.concat([real_df, fake_df]).reset_index(drop=True)

MAX_SAMPLES = len(sample_df)
print(f"해시 계산 대상: {len(sample_df):,}장 (Real {len(real_df):,} + AI {len(fake_df):,})")

img_dir_map = {
    f"shard_{i}": os.path.join(SHARD_ROOT, f"shard_{i}", "images")
    for i in SHARD_NUMS
}

records = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="해시 계산"):
    img_path = os.path.join(img_dir_map[row["shard_name"]], row["image_name"])
    try:
        img   = Image.open(img_path).convert("RGB")
        dhash = str(imagehash.dhash(img))
        phash = str(imagehash.phash(img))
        records.append({
            "shard_name": row["shard_name"],
            "image_name": row["image_name"],
            "label":      int(row["label"]),
            "dhash":      dhash,
            "phash":      phash,
        })
    except Exception as e:
        print(f"오류: {img_path} — {e}")

hash_df = pd.DataFrame(records)
hash_df.to_csv("../results/hashes.csv", index=False)
print(f"\n해시 계산 완료: {len(hash_df):,}장")
print(hash_df[["dhash","phash"]].head())


In [ ]:
# ============================================================
# Train / Test 간 중복 탐지
# 8:1:1 분할 기준으로 Train(8000) / Test(1000) 분리 후 비교
# ============================================================
import torch
from torch.utils.data import random_split

total      = len(hash_df)
train_size = int(0.8 * total)  # 8000
val_size   = int(0.1 * total)  # 1000
test_size  = total - train_size - val_size  # 1000

# 동일한 seed로 분할 (02_train.ipynb 와 동일하게)
generator = torch.Generator().manual_seed(42)
indices   = list(range(total))
train_idx, val_idx, test_idx = random_split(
    indices, [train_size, val_size, test_size], generator=generator
)

train_hashes = set(hash_df.iloc[list(train_idx)]["dhash"].tolist())
test_hashes  = set(hash_df.iloc[list(test_idx)]["dhash"].tolist())

# Train-Test 간 dHash 겹치는 것 탐지
duplicates = train_hashes & test_hashes

print(f"Train 이미지 수 : {len(train_hashes):,}")
print(f"Test  이미지 수 : {len(test_hashes):,}")
print(f"중복 dHash 수   : {len(duplicates)}")

if len(duplicates) == 0:
    print("\n✅ Train/Test 간 중복 없음 — 데이터 누수 없음")
else:
    print(f"\n⚠️  중복 발견! 아래 정리 셀에서 제거 필요")

In [ ]:
# ============================================================
# 내부 중복 탐지 (같은 이미지가 여러 번 포함된 경우)
# ============================================================
# dHash 기준 내부 중복
dhash_duplicates = hash_df[hash_df.duplicated(subset="dhash", keep=False)]
phash_duplicates = hash_df[hash_df.duplicated(subset="phash", keep=False)]

print(f"dHash 내부 중복: {len(dhash_duplicates)}장")
print(f"pHash 내부 중복: {len(phash_duplicates)}장")

if len(dhash_duplicates) > 0:
    print("\n중복 샘플 예시:")
    print(dhash_duplicates[["image_name", "label", "dhash"]].head(10))

In [ ]:
# ============================================================
# 중복 제거 후 shard별 clean_labels.csv 저장
# ============================================================
clean_df = hash_df.drop_duplicates(subset="dhash", keep="first")

removed = len(hash_df) - len(clean_df)
print(f"원본: {len(hash_df):,}장")
print(f"제거: {removed}장")
print(f"정제 후: {len(clean_df):,}장")
print(f"\n클래스 분포:")
print(clean_df["label"].value_counts().rename({0: "Real", 1: "AI Generated"}))

# shard별로 분리해서 각각 저장
for i in SHARD_NUMS:
    shard_clean = clean_df[clean_df["shard_name"] == f"shard_{i}"][["image_name", "label"]]
    out_path = os.path.join(SHARD_ROOT, f"shard_{i}", "labels_clean.csv")
    shard_clean.to_csv(out_path, index=False)
    print(f"shard_{i} 정제 완료: {len(shard_clean):,}장 → {out_path}")


In [ ]:
# ============================================================
# 중복 이미지 시각화 (중복이 있을 경우)
# ============================================================
if len(dhash_duplicates) > 0:
    first_hash = dhash_duplicates["dhash"].iloc[0]
    group      = dhash_duplicates[dhash_duplicates["dhash"] == first_hash]

    fig, axes = plt.subplots(1, len(group), figsize=(5 * len(group), 5))
    if len(group) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, group.iterrows()):
        img_path = os.path.join(img_dir_map[row["shard_name"]], row["image_name"])
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{'Real' if row['label']==0 else 'AI'}\n{row['image_name'][:15]}...")
        ax.axis("off")

    plt.suptitle(f"중복 이미지 예시 (dHash: {first_hash})", fontsize=12)
    plt.tight_layout()
    plt.savefig("../results/duplicate_examples.png", dpi=150)
    plt.show()
else:
    print("중복 이미지 없음 — 시각화 생략")


In [ ]:
# ============================================================
# 최종 요약 리포트
# ============================================================
print("=" * 50)
print("Deduplication 결과 요약")
print("=" * 50)
print(f"대상 이미지       : {len(hash_df):,}장")
print(f"내부 중복 (dHash) : {removed}장 제거")
print(f"Train-Test 중복   : {len(duplicates)}건")
print(f"정제 후 최종 수   : {len(clean_df):,}장")
print(f"데이터 누수 위험  : {'⚠️  있음' if len(duplicates) > 0 else '✅ 없음'}")
print("=" * 50)